
# Desafio 1 — Exploração Inicial da Base

## Objetivo
Conhecer a estrutura da base de dados do formulário sobre o uso de IA no ICH/UFJF, identificando suas dimensões, tipos de dados e valores ausentes.

## Análise
Utilizei a biblioteca Pandas e os seguintes métodos:

- `pd.read_csv()` — carreguei o arquivo CSV.
- `df.head()` — visualizei as primeiras respostas.
- `df.shape` — identifiquei a dimensão da base.
- `df.info()` — verifiquei os tipos de dados.
- `df.isnull().sum()` — contei os valores ausentes.

## Resultados
- **72 respostas e 18 colunas.**
- Todas as colunas foram inicialmente interpretadas como texto (`str`).
- Identifiquei valores ausentes em algumas perguntas, principalmente nas destinadas a perfis específicos e nas questões abertas.

## Conclusão
Consegui compreender a estrutura inicial da base e identificar pontos que precisam de investigação antes do tratamento dos dados.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/ia.csv')

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()


# Desafio 2 — Perfil dos Respondentes

## Objetivo
Identificar os perfis dos participantes e investigar a relação entre o enquadramento e os valores ausentes.

## Análise
Utilizei os métodos:

- `unique()` — identifiquei as categorias de enquadramento.
- `value_counts()` — contei os participantes de cada categoria.
- `isnull()` — investiguei os valores ausentes.

## Resultados

| Enquadramento             | Quantidade |
| Docente                   | 68         |
| Técnico-Administrativo    | 3          |
| Discente de Pós-Graduação | 1          |
| **Total**                 | **72**     |

Identifiquei:
- **71 valores ausentes** na pergunta destinada aos discentes.
- **4 valores ausentes** na pergunta destinada aos docentes.

Essas quantidades correspondem aos participantes que não pertencem aos respectivos perfis.

## Conclusão
Identifiquei que os docentes representam 94,4% dos participantes, os Técnico-Administrativo representam 4.17% e o Discente de Pós-Graduação (Mestrado/Doutorado) representa 1.39%. Também observei que os valores ausentes nessas perguntas são compatíveis com a lógica do formulário, indicando possíveis ausências estruturais, ainda a confirmar por filtragem.


In [ ]:
df['Enquadramento'].unique()

In [ ]:
df['Enquadramento'].value_counts()

In [ ]:
(df['Enquadramento'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)


# Desafio 3 — Identificação de Duplicatas e Registros Suspeitos

## Objetivo
Investigar possíveis duplicatas e respostas de teste na base de dados do formulário do ICH/UFJF.

## Análise
Utilizei os seguintes métodos:

- `duplicated()` — identifiquei e-mails repetidos e duplicatas completas.
- `iloc[]` — comparei duas submissões do mesmo participante.
- `str.contains()` — procurei respostas contendo "Teste".
- `apply()` e `any(axis=1)` — ampliei a busca para todas as colunas.
- `sum()` — contei os registros encontrados.

## Resultados
- Identifiquei 2 submissões vinculadas ao mesmo e-mail.
- Não encontrei linhas completamente duplicadas.
- Localizei 1 registro contendo "Teste", no índice 1.
- Nesse registro, encontrei 6 campos preenchidos exatamente com "Teste".

## Conclusão
Identifiquei uma provável submissão de teste no índice 1, que também está associada a um e-mail repetido.

Considerei esse registro suspeito e recomendo validar sua exclusão com o professor antes de realizar a limpeza definitiva. Mantive a base original sem alterações.


In [ ]:
df['Endereço de e-mail'].duplicated(keep=False).sum()

In [ ]:
df.duplicated().sum()

In [ ]:

quantidade_testes = (
    df.astype('string')
    .apply(
        lambda coluna: coluna.str.strip().str.casefold().eq('teste')
    )
    .fillna(False)
    .sum(axis=1)
)

print("Registros suspeitos:", (quantidade_testes >= 5).sum())
print("Maior quantidade de campos com Teste:", quantidade_testes.max())


# Desafio 4 — Limpeza e Padronização dos Dados

## Objetivo
Limpar, padronizar e organizar as respostas do formulário do ICH/UFJF, preparando os dados para a análise exploratória e o desenvolvimento do dashboard.

## Análise
Utilizei a biblioteca Pandas e os seguintes métodos:

- `copy()` — criei uma cópia da base original.
- `apply()`, `str.strip()` e `str.casefold()` — identifiquei o registro de teste pelo conteúdo.
- `drop()` e `reset_index()` — removi o registro inválido e reorganizei os índices.
- `rename()` — padronizei os nomes das colunas.
- `pd.to_datetime()` — converti as datas das respostas.
- `replace()` e `map()` — categorizei e padronizei respostas.
- `str.contains()` — identifiquei alternativas nas perguntas de múltipla escolha.
- `value_counts()` e `sum()` — calculei frequências e quantidades.
- `equals()` e `assert()` — verifiquei a consistência das classificações.
- `json.load()` — carreguei os mapeamentos salvos para reutilização.

## Resultados
- A base original possuía **72 respostas e 18 colunas**.
- Identifiquei e removi **1 registro de teste**, permanecendo **71 respostas válidas**.
- Padronizei os nomes das colunas e converti os registros de data.
- Criei **7 mapeamentos de classificação**, incluindo orientações docentes, práticas de ensino, Portaria CNPq e sigilo de documentos.
- Separei as alternativas das perguntas de múltipla escolha em indicadores individuais.
- Identifiquei e classifiquei **8 respostas personalizadas** sobre usos de IA na pesquisa.
- Identifiquei e classifiquei **5 respostas personalizadas** sobre riscos na gestão.
- Não encontrei contradições entre as alternativas oficiais analisadas.
- As comparações dos indicadores apresentaram **0 divergências**.
- Confirmei a integridade dos **71 registros** e a correspondência dos índices.
- Armazenei os mapeamentos em um arquivo JSON local, permitindo reutilizar as classificações sem depender dos índices das respostas.

## Conclusão
Concluí a limpeza e a padronização da base, removendo o registro de teste e organizando as respostas em categorias consistentes.

Validei os indicadores, preservei os dados originais e organizei o processamento para permitir sua reprodução. Com isso, preparei a base para a análise exploratória e a construção do dashboard do ICH/UFJF.

In [ ]:
import json
from pathlib import Path

raiz = Path.cwd()

if raiz.name.lower() == 'notebooks':
    raiz = raiz.parent

with open(raiz / 'data' / 'mapeamentos_categorias.json',
          'r', encoding='utf-8') as arquivo:
    mapeamentos = json.load(arquivo)

mapeamento_docente = mapeamentos['orientacao_docente']
mapeamento_praticas = mapeamentos['praticas_ensino']
mapeamento_sigilo = mapeamentos['sigilo_documentos']
mapeamento_portaria = mapeamentos['portaria_cnpq']
mapeamento_opcoes = mapeamentos['praticas_opcoes_mencionadas']
mapeamento_outros_pesquisa = mapeamentos['outros_uso_pesquisa']
mapeamento_outros_riscos = mapeamentos['outros_riscos_gestao']

In [ ]:

df_limpo = df.copy()

quantidade_teste = (
    df_limpo.astype('string')
    .apply(
        lambda coluna: coluna.str.strip().str.casefold().eq('teste')
    )
    .fillna(False)
    .sum(axis=1)
)

registros_teste = quantidade_teste >= 5

print("Registros de teste identificados:", registros_teste.sum())

assert registros_teste.sum() == 1, (
    "Quantidade inesperada de registros de teste. Revisar antes de excluir."
)

df_limpo = df_limpo.loc[~registros_teste].copy()

print("Registros restantes:", len(df_limpo))


In [ ]:
df_limpo.shape

In [ ]:
df_limpo = df_limpo.reset_index(drop=True)

In [ ]:
df_limpo.index

In [ ]:
df_limpo.columns

In [ ]:
df_limpo.columns = ['data_resposta', 'email', 'fe_enquadramento', 'fe_orientacao_docente','fe_orientacao_discente', 'fe_departamento', 
                    'multi_uso_ia_pesquisa', 'fe_opiniao_portaria_cnpq', 'ab_riscos_pesquisa', 'ab_beneficios_pesquisa', 'fe_familiaridade_ia', 
                    'fe_praticas_ensino_ia', 'ab_preparacao_uso_etico', 'multi_riscos_gestao', 'fe_sigilo_documentos', 'ab_diretrizes_transparencia', 
                    'ab_comentarios_politica', 'ab_outros_topicos']

In [ ]:
df_limpo.columns

In [ ]:
df.columns

In [ ]:
df_limpo.info()

In [ ]:
df_limpo['data_resposta'] = pd.to_datetime(
    df_limpo['data_resposta'],
    format='%d/%m/%Y %H:%M:%S'
)

In [ ]:
df_limpo['data_resposta'].dtype

In [ ]:
df_limpo.info()

In [ ]:
df_limpo['fe_enquadramento'].unique()

In [ ]:
df_limpo['fe_enquadramento'].value_counts(dropna=False)

In [ ]:
df_limpo['fe_familiaridade_ia'].unique()

In [ ]:
df_limpo['fe_familiaridade_ia'].value_counts(dropna=False)

In [ ]:
(df_limpo['fe_enquadramento'] != df_limpo['fe_enquadramento'].str.strip()).sum()

In [ ]:
(df_limpo['fe_familiaridade_ia'] != df_limpo['fe_familiaridade_ia'].str.strip()).sum()

In [ ]:
colunas_fechadas = [
    'fe_enquadramento',
    'fe_orientacao_docente',
    'fe_orientacao_discente',
    'fe_familiaridade_ia',
    'fe_praticas_ensino_ia',
    'fe_sigilo_documentos'
]

for coluna in colunas_fechadas:
    print('Coluna:', coluna)
    print('Categorias distintas:', df_limpo[coluna].nunique())
    print('Valores ausentes:', df_limpo[coluna].isnull().sum())
    print('--------------------')

In [ ]:
df_limpo['fe_familiaridade_ia'].value_counts(dropna=False).head(10)

In [ ]:
opcoes_pesquisa = {
    'levantamento_bibliografico': 'Levantamento bibliográfico',
    'revisao_bibliografica': 'Revisão bibliográfica',
    'sintese_textos': 'Síntese de textos',
    'analise_dados': 'Análise de dados (qualitativos ou quantitativos)',
    'traducao_textos': 'Tradução de textos',
    'revisao_textos': 'Revisão de textos',
    'geracao_ideias': 'Geração de ideias ou estruturação de argumentos',
    'redacao_parcial': 'Redação parcial de seções (ex: introdução, metodologia, conclusão...)',
    'redacao_total': 'Redação total de seções (ex: introdução, metodologia, conclusão...)',
    'transcricao': 'Transcrição',
    'nunca_utilizei': 'Nunca utilizei',
    'nao_se_aplica': 'Não se aplica a mim'
}


In [ ]:

df_uso = pd.DataFrame(index=df_limpo.index)

for nome, alternativa in opcoes_pesquisa.items():
    df_uso[nome] = (
        df_limpo['multi_uso_ia_pesquisa']
        .str.contains(alternativa, regex=False, na=False)
    )



In [ ]:

frequencia_uso = df_uso.sum().sort_values(ascending=False)

frequencia_uso


In [ ]:
import re

padrao_pesquisa = '|'.join(
    re.escape(opcao)
    for opcao in sorted(
        opcoes_pesquisa.values(),
        key=len,
        reverse=True
    )
)
restos_pesquisa = (
    df_limpo['multi_uso_ia_pesquisa']
    .fillna('')
    .str.replace(padrao_pesquisa, '', regex=True)
    .str.strip(' ,;')
)
outros_pesquisa = df_limpo.loc[
    restos_pesquisa.ne(''),
    ['multi_uso_ia_pesquisa']
].copy()
outros_pesquisa['texto_outro'] = restos_pesquisa[
    restos_pesquisa.ne('')
]
print('Respostas personalizadas:', len(outros_pesquisa))


In [ ]:
opcoes_riscos = {
    'violacoes_sigilo': 'Violações de sigilo',
    'informacoes_enviesadas': 'Informações enviesadas',
    'precarizacao_trabalho': 'Precarização do trabalho humano',
    'sem_riscos': 'Não vejo riscos significativos'
}

import re

df_riscos = pd.DataFrame(index=df_limpo.index)

for nome, alternativa in opcoes_riscos.items():
    padrao = rf'(?:^|,\s*){re.escape(alternativa)}(?=,|$)'

    df_riscos[nome] = (
        df_limpo['multi_riscos_gestao']
        .str.contains(padrao, regex=True, na=False)
    )



In [ ]:
frequencia_riscos = (
    df_riscos.sum()
    .sort_values(ascending=False)
)

frequencia_riscos

In [ ]:
padrao_riscos = '|'.join(
    re.escape(opcao)
    for opcao in sorted(
        opcoes_riscos.values(),
        key=len,
        reverse=True
    )
)

restos_riscos = (
    df_limpo['multi_riscos_gestao']
    .fillna('')
    .str.replace(padrao_riscos, '', regex=True)
    .str.strip(' ,;')
)

outros_riscos = df_limpo.loc[
    restos_riscos.ne(''),
    ['multi_riscos_gestao']
].copy()

outros_riscos['texto_outro'] = restos_riscos[
    restos_riscos.ne('')
]

print('Respostas personalizadas:', len(outros_riscos))


In [ ]:
df_limpo['portaria_cnpq_categoria'] = (
    df_limpo['fe_opiniao_portaria_cnpq']
    .replace(mapeamento_portaria)
)

df_limpo['sigilo_categoria'] = (
    df_limpo['fe_sigilo_documentos']
    .replace(mapeamento_sigilo)
)

df_limpo['orientacao_docente_categoria'] = (
    df_limpo['fe_orientacao_docente']
    .replace(mapeamento_docente)
)

df_limpo['praticas_ensino_categoria'] = (
    df_limpo['fe_praticas_ensino_ia']
    .str.strip()
    .replace(mapeamento_praticas)
)

df_limpo['praticas_ensino_opcoes_mencionadas'] = (
    df_limpo['fe_praticas_ensino_ia']
    .map(mapeamento_opcoes)
    .astype('string')
)

df_limpo['uso_ia_outro_categoria'] = (
    restos_pesquisa.map(mapeamento_outros_pesquisa)
)

df_limpo['riscos_outro_categoria'] = (
    restos_riscos.map(mapeamento_outros_riscos)
)

print('Os 7 mapeamentos foram aplicados!')
print('Registros:', len(df_limpo))

In [ ]:
esperado = {
    'portaria_cnpq_categoria': 68,
    'sigilo_categoria': 71,
    'orientacao_docente_categoria': 67,
    'praticas_ensino_categoria': 71,
    'praticas_ensino_opcoes_mencionadas': 7,
    'uso_ia_outro_categoria': 8,
    'riscos_outro_categoria': 5
}

assert len(df_limpo) == 71

for coluna, quantidade in esperado.items():
    preenchidos = df_limpo[coluna].notna().sum()
    assert preenchidos == quantidade, f'Erro em {coluna}'
    print(f'{coluna}: {preenchidos} preenchidos')

assert df_limpo.index.equals(df_uso.index)
assert df_limpo.index.equals(df_riscos.index)

print('\nTodas as validações passaram!')

In [ ]:
nao_usa = (
    df_uso['nunca_utilizei'] |
    df_uso['nao_se_aplica']
)

usos_positivos = df_uso.drop(
    columns=['nunca_utilizei', 'nao_se_aplica']
).any(axis=1)

conflitos_uso = nao_usa & usos_positivos

print(
    "Possíveis contradições no uso de IA:",
    conflitos_uso.sum()
)


In [ ]:
sem_riscos = df_riscos['sem_riscos']

riscos_identificados = df_riscos.drop(
    columns=['sem_riscos']
).any(axis=1)

conflitos_riscos = sem_riscos & riscos_identificados

print(
    "Possíveis contradições nos riscos:",
    conflitos_riscos.sum()
)


In [ ]:

print("Registros originais limpos:", len(df_limpo))
print("Registros dos indicadores de uso:", len(df_uso))
print("Registros dos indicadores de riscos:", len(df_riscos))

print(
    "Índices de uso correspondem:",
    df_limpo.index.equals(df_uso.index)
)

print(
    "Índices de riscos correspondem:",
    df_limpo.index.equals(df_riscos.index)
)


In [ ]:
df_uso_validado = pd.DataFrame(index=df_limpo.index)

for nome, alternativa in opcoes_pesquisa.items():

    padrao = rf'(?:^|,\s*){re.escape(alternativa)}(?=,\s*|$)'

    df_uso_validado[nome] = (
        df_limpo['multi_uso_ia_pesquisa']
        .str.contains(padrao, regex=True, na=False)
    )

divergencias_uso = df_uso != df_uso_validado

print(
    'Indicadores diferentes:',
    divergencias_uso.to_numpy().sum()
)

print(
    'Participantes com alguma diferença:',
    divergencias_uso.any(axis=1).sum()
)

print(
    'Índices para revisão:',
    df_limpo.index[divergencias_uso.any(axis=1)].tolist()
)


In [ ]:
colunas_analise = [
    'fe_enquadramento',
    'fe_familiaridade_ia',
    'orientacao_docente_categoria',
    'praticas_ensino_categoria',
    'portaria_cnpq_categoria',
    'sigilo_categoria'
]

df_indicadores = pd.concat(
    [
        df_limpo[colunas_analise],
        df_uso.add_prefix('uso_'),
        df_riscos.add_prefix('riscos_')
    ],
    axis=1
)

# ATUALIZACAO DESAFIO 5: OPCOES OUTROS
# Indicadores booleanos: 1 registro = 1 participante, mesmo que cite vários temas.
df_indicadores['uso_outros'] = df_limpo['uso_ia_outro_categoria'].notna()
df_indicadores['riscos_outros'] = df_limpo['riscos_outro_categoria'].notna()

assert df_indicadores.shape == (71, 24), 'Esperadas 71 linhas e 24 colunas'
assert int(df_indicadores['uso_outros'].sum()) == 8, 'Revisar usos em Outros'
assert int(df_indicadores['riscos_outros'].sum()) == 5, 'Revisar riscos em Outros'

# As verificações anteriores de alternativas fechadas permanecem válidas.
# Conferimos também possíveis conflitos entre 'Outros' e opções de ausência.
conflitos_uso_outros = int((
    df_indicadores['uso_outros'] & df_indicadores['uso_nunca_utilizei']
).sum())
conflitos_riscos_outros = int((
    df_indicadores['riscos_outros'] & df_indicadores['riscos_sem_riscos']
).sum())
print('Conflitos de Outros com Nunca utilizei:', conflitos_uso_outros)
print('Conflitos de Outros com Sem riscos:', conflitos_riscos_outros)
if conflitos_uso_outros or conflitos_riscos_outros:
    print('ATENÇÃO: revisar essas combinações localmente antes de interpretar.')



print("Dimensões:", df_indicadores.shape)
print("Total de registros:", len(df_indicadores))
print("Total de colunas:", len(df_indicadores.columns))

In [ ]:
assert len(df_indicadores) == 71
assert df_indicadores.index.equals(df_limpo.index)

caminho_indicadores = raiz / 'data' / 'indicadores_ich.csv'

df_indicadores.to_csv(
    caminho_indicadores,
    index=False,
    encoding='utf-8-sig'
)

print("Base analítica salva com sucesso!")
print("Arquivo encontrado:", caminho_indicadores.exists())

In [ ]:
categorias_outros_riscos = pd.Series(
    list(mapeamento_outros_riscos.values())
)

print("Categorias encontradas em Outros:")
print(categorias_outros_riscos.value_counts())

print("\nTotal de respostas categorizadas:",
      len(categorias_outros_riscos))

In [ ]:
categorias_outros_usos = pd.Series(
    list(mapeamento_outros_pesquisa.values())
)

print("Categorias de Outros — Usos de IA:")
print(categorias_outros_usos.value_counts())

print("\nTotal de entradas mapeadas:",
      len(categorias_outros_usos))

In [ ]:
for coluna in df_limpo.columns:
    if 'outros' in coluna.lower() or 'categoria' in coluna.lower():
        print(coluna)

In [ ]:
outros_usos = df_limpo['uso_ia_outro_categoria'].dropna()

print("Total de respostas em Outros:", len(outros_usos))

print("\nCategorias encontradas:")
print(outros_usos.value_counts())

In [ ]:
temas_outros_usos = (
    outros_usos
    .str.split(';')
    .explode()
    .str.strip()
    .value_counts()
)

print("Frequência dos temas de Outros:")
print(temas_outros_usos)

print("\nTotal de respostas:", len(outros_usos))
print("Total de menções:", temas_outros_usos.sum())

In [ ]:
agrupamentos = {
    'Produção visual': 'Produção visual e audiovisual',
    'Produção visual e audiovisual': 'Produção visual e audiovisual',

    'Formatação': 'Formatação',
    'Formatação de projetos': 'Formatação',
    'Formatação de textos': 'Formatação'
}

temas_agrupados = (
    temas_outros_usos
    .rename(index=agrupamentos)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("Temas agrupados:")
print(temas_agrupados)

print("\nTotal de menções:", temas_agrupados.sum())

In [ ]:
tabela_outros_usos = (
    temas_agrupados
    .rename_axis('Tema')
    .reset_index(name='Menções')
)

print("Tabela de outros usos de IA:")
print(tabela_outros_usos)

print("\nTotal de menções:", tabela_outros_usos['Menções'].sum())
print("Total de participantes com respostas em Outros:", len(outros_usos))

In [ ]:
from pathlib import Path

tabela_outros_usos = (
    temas_agrupados
    .rename_axis('Tema')
    .reset_index(name='Menções')
)

raiz = Path.cwd()

if raiz.name.lower() == 'notebooks':
    raiz = raiz.parent

caminho_saida = raiz / 'data' / 'resumo_outros_usos.csv'

tabela_outros_usos.to_csv(
    caminho_saida,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo salvo em:", caminho_saida)
print("Total de temas:", len(tabela_outros_usos))
print("Total de menções:", tabela_outros_usos['Menções'].sum())

In [ ]:
# ATUALIZACAO DESAFIO 5: RESUMO DE OUTROS RISCOS
# Usamos apenas classificações; nenhuma resposta original ou e-mail é exportado.
outros_riscos = df_limpo['riscos_outro_categoria'].dropna()

temas_outros_riscos = (
    outros_riscos.astype('string')
    .str.split(';')
    .explode()
    .str.strip()
)
temas_outros_riscos = temas_outros_riscos[
    temas_outros_riscos.notna() & temas_outros_riscos.ne('')
]

tabela_outros_riscos = (
    temas_outros_riscos.value_counts()
    .rename_axis('Tema')
    .reset_index(name='Menções')
)

assert len(outros_riscos) == 5, 'Esperadas 5 respostas categorizadas em Outros riscos'
assert tabela_outros_riscos['Menções'].sum() >= len(outros_riscos)

saida_riscos = raiz / 'data' / 'resumo_outros_riscos.csv'
tabela_outros_riscos.to_csv(saida_riscos, index=False, encoding='utf-8-sig')
print('Resumo agregado de Outros riscos salvo em data/resumo_outros_riscos.csv')
print('Respostas com Outros:', len(outros_riscos))
print('Menções categorizadas:', int(tabela_outros_riscos['Menções'].sum()))

In [ ]:
opcoes_originais_docentes = [
    'Há uma política clara e formalizada no meu curso/departamento',
    'Orientações informais, combinadas entre docente e discentes',
    'O uso é proibido por regra geral',
    'O uso é livre, sem qualquer orientação específica',
    'O tema nunca foi tratado'
]

respostas_docentes = (
    df_limpo['fe_orientacao_docente']
    .dropna()
    .str.strip()
)

marcou_original = respostas_docentes.isin(opcoes_originais_docentes)

print("Alternativas originais:", marcou_original.sum())
print("Respostas de Outros:", (~marcou_original).sum())
print("Total de respostas:", len(respostas_docentes))

In [ ]:
indices_divergentes = respostas_docentes[
    ~marcou_original
].index

categorias_divergentes = df_limpo.loc[
    indices_divergentes,
    'orientacao_docente_categoria'
]

categorias_originais_docentes = [
    'Há uma política clara e formalizada no meu curso/departamento',
    'Orientações informais',
    'O uso é proibido por regra geral',
    'O uso é livre, sem qualquer orientação específica',
    'O tema nunca foi tratado'
]

print("Categorias das 11 respostas:")
print(categorias_divergentes.value_counts())

print("\nRespostas classificadas como alternativas originais:")
print(
    categorias_divergentes.isin(categorias_originais_docentes).sum()
)

In [ ]:
registro_divergente = df_limpo.loc[
    indices_divergentes[
        categorias_divergentes == 'Orientações informais'
    ],
    ['fe_orientacao_docente', 'orientacao_docente_categoria']
]

print("Resposta original e classificação:")
print(registro_divergente.to_string(index=False))

In [ ]:
colunas_praticas = [
    coluna
    for coluna in df_limpo.columns
    if 'pratic' in coluna.lower()
    or 'ensino' in coluna.lower()
]

print("Colunas relacionadas às práticas de ensino:")

for coluna in colunas_praticas:
    print(coluna)

In [ ]:
print("Opções mencionadas nas respostas personalizadas:")

print(
    df_limpo['praticas_ensino_opcoes_mencionadas']
    .value_counts(dropna=False)
)

print("\nTotal de registros com opções identificadas:")

print(
    df_limpo['praticas_ensino_opcoes_mencionadas']
    .notna()
    .sum()
)

In [ ]:
frequencia_opcoes = (
    df_limpo['praticas_ensino_opcoes_mencionadas']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .sort_index()
)

print("Frequência de cada opção mencionada:")
print(frequencia_opcoes)

print("\nTotal de respostas personalizadas:", 7)
print("Total de menções:", frequencia_opcoes.sum())

In [ ]:
filtro_multiplas = (
    df_limpo['praticas_ensino_opcoes_mencionadas'].notna()
)

print("Classificação das 7 respostas:")

print(
    df_limpo.loc[
        filtro_multiplas,
        'praticas_ensino_categoria'
    ].value_counts()
)

print("\nTotal de registros:", filtro_multiplas.sum())

In [ ]:
inicios_originais = (
    'Estimular o uso crítico e transparente',
    'Proibir integralmente o uso em atividades avaliativas',
    'Incorporar a IA como conteúdo a ser estudado',
    'Utilizar a IA como ferramenta auxiliar docente'
)

respostas_praticas = (
    df_limpo['fe_praticas_ensino_ia']
    .fillna('')
    .str.strip()
)

eh_original = respostas_praticas.str.startswith(inicios_originais)

print("Respostas que correspondem às alternativas originais:",
      eh_original.sum())

print("Respostas que não coincidem com as alternativas:",
      (~eh_original).sum())

print("\nClassificações das respostas que não coincidiram:")
print(
    df_limpo.loc[
        ~eh_original,
        'praticas_ensino_categoria'
    ].value_counts()
)



In [ ]:
filtro_multiplas = (
    df_limpo['praticas_ensino_categoria']
    == 'Preferência por múltiplas práticas'
)

print("Total de respostas com múltiplas práticas:")
print(filtro_multiplas.sum())

print("\nRespostas com múltiplas práticas que começam com uma alternativa original:")
print((filtro_multiplas & eh_original).sum())

print("\nRespostas com múltiplas práticas que não começam com uma alternativa original:")
print((filtro_multiplas & ~eh_original).sum())

In [ ]:
categorias_principais_praticas = [
    'Uso crítico e transparente',
    'Restrição de IA em avaliações',
    'IA como conteúdo de estudo',
    'IA como ferramenta auxiliar docente'
]

eh_categoria_principal = (
    df_limpo['praticas_ensino_categoria']
    .isin(categorias_principais_praticas)
)

tabela_validacao = pd.crosstab(
    eh_categoria_principal,
    eh_original,
    rownames=['Categoria principal?'],
    colnames=['Começa como alternativa original?']
)

display(tabela_validacao)

In [ ]:
respostas_etica = (
    df_limpo['ab_preparacao_uso_etico']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

print("Total de participantes:", len(respostas_etica))
print("Respostas ausentes:", respostas_etica.isna().sum())
print("Respostas presentes:", respostas_etica.notna().sum())



In [ ]:
tamanhos_etica = respostas_etica.dropna().str.len()
print("\nTamanho médio:",
      round(tamanhos_etica.mean(), 1),
      "caracteres")

print("Menor resposta:",
      tamanhos_etica.min(),
      "caracteres")

print("Maior resposta:",
      tamanhos_etica.max(),
      "caracteres")

In [ ]:
analise_etica = pd.DataFrame({
    'ID': range(1, len(respostas_etica) + 1),
    'Resposta': respostas_etica.values
})

analise_etica['Categorias'] = pd.NA

print("Total de respostas para análise:", len(analise_etica))
print("Colunas:", analise_etica.columns.tolist())

display(analise_etica[['ID', 'Categorias']].head())

In [ ]:
categorias_etica = [
    'Capacitação e formação',
    'Materiais de orientação',
    'Diretrizes institucionais',
    'Comunicação e debate',
    'Pensamento crítico',
    'Integridade acadêmica',
    'Proteção e verificação',
    'Acesso e infraestrutura tecnológica',
    'Integração curricular',
    'Adaptação pedagógica e avaliações',
    'Fiscalização e controle',
    'Participação democrática',
    'Oposição ao uso de IA',
    'Ceticismo institucional',
    'Sem opinião ou sugestão'
]

print("Total de categorias:", len(categorias_etica))

for numero, categoria in enumerate(categorias_etica, start=1):
    print(f"{numero}. {categoria}")

In [ ]:
codigos_etica = {
    1: [1, 2],
    2: [4],
    3: [5],
    4: [2, 8],
    5: [1, 5, 6],
    6: [1],
    7: [1, 3],
    8: [1, 3, 4, 5, 6, 7],
    9: [1, 2, 3],
    10: [1, 4, 9],

    11: [1, 4, 6],
    12: [3],
    13: [1, 2, 6],
    14: [14],
    15: [1, 3, 5],
    16: [1, 4, 5, 6],
    17: [1, 4],
    18: [4, 6],
    19: [3, 9],
    20: [1, 3, 4],

    21: [15],
    22: [4],
    23: [5, 13],
    24: [3, 12],
    25: [4, 5, 6, 9],
    26: [1, 4],
    27: [1, 4, 5],
    28: [14],
    29: [3, 4, 5],
    30: [1],

    31: [2],
    32: [1, 2],
    33: [1, 5, 6, 10],
    34: [1, 9],
    35: [2, 5, 10],
    36: [1, 3],
    37: [1, 2, 4],
    38: [1, 3],
    39: [3, 6, 9, 11],
    40: [1],

    41: [1, 2],
    42: [15],
    43: [1, 8, 9],
    44: [3, 6, 7],
    45: [1, 6],
    46: [1, 5],
    47: [3, 4],
    48: [15],
    49: [3, 5, 9],
    50: [1],

    51: [15],
    52: [1, 6],
    53: [1, 11],
    54: [1, 11],
    55: [1, 4, 9],
    56: [1],
    57: [1, 3, 6],
    58: [1, 3, 5],
    59: [1, 4, 5],
    60: [1, 3, 5, 6],

    61: [1, 3, 4],
    62: [15],
    63: [2, 5, 8, 9],
    64: [3],
    65: [4, 5, 6, 9],
    66: [1, 5],
    67: [1],
    68: [1],
    69: [1, 2, 3, 7],
    70: [1, 3, 4, 12],
    71: [2]
}

In [ ]:
arquivo_saida = raiz / 'data' / 'categorias_etica.csv'

analise_etica[['ID', 'Categorias']].assign(
    Categorias=lambda df: df['Categorias'].apply(
        lambda categorias: '|'.join(categorias)
    )
).to_csv(
    arquivo_saida,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo criado:", arquivo_saida.name)
print("Total de respostas:", len(analise_etica))

In [ ]:
palavras_chave = [
    'departamento',
    'curso',
    'programa'
]

colunas_encontradas = [
    coluna
    for coluna in df_limpo.columns
    if any(
        palavra in coluna.lower()
        for palavra in palavras_chave
    )
]

print("Colunas encontradas:")
print(colunas_encontradas)

for coluna in colunas_encontradas:

    print(f"\nColuna: {coluna}")

    print(
        "Respostas presentes:",
        df_limpo[coluna].notna().sum()
    )

    print(
        "Respostas ausentes:",
        df_limpo[coluna].isna().sum()
    )

    print("\nCategorias existentes:")
    print(
        df_limpo[coluna].value_counts(
            dropna=False
        )
    )

In [ ]:
assert len(df_limpo) == len(df_indicadores) == 71
assert df_limpo.index.equals(df_indicadores.index)

print("Perfil dos participantes sem departamento:")

print(
    df_limpo.loc[
        df_limpo['fe_departamento'].isna(),
        'fe_enquadramento'
    ].value_counts(dropna=False)
)

df_indicadores['fe_departamento'] = (
    df_limpo['fe_departamento']
)

print("\nTotal de participantes:", len(df_indicadores))
print("Total de colunas:", len(df_indicadores.columns))

print(
    "Departamentos preenchidos:",
    df_indicadores['fe_departamento'].notna().sum()
)

print(
    "Departamentos ausentes:",
    df_indicadores['fe_departamento'].isna().sum()
)

In [ ]:
arquivo_indicadores = raiz / 'data' / 'indicadores_ich.csv'

df_indicadores.to_csv(
    arquivo_indicadores,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo atualizado:", arquivo_indicadores.name)
print("Formato:", df_indicadores.shape)

In [ ]:

respostas_riscos_pesquisa = (
    df_limpo['ab_riscos_pesquisa']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

print("Total de participantes:", len(df_limpo))

print(
    "Respostas presentes:",
    respostas_riscos_pesquisa.notna().sum()
)

print(
    "Respostas ausentes:",
    respostas_riscos_pesquisa.isna().sum()
)

tamanhos_riscos = (
    respostas_riscos_pesquisa
    .dropna()
    .str.len()
)

print("\nTamanho médio das respostas:",
      round(tamanhos_riscos.mean(), 1)
      if len(tamanhos_riscos) else 0)

print("Menor resposta:",
      tamanhos_riscos.min()
      if len(tamanhos_riscos) else 0)

print("Maior resposta:",
      tamanhos_riscos.max()
      if len(tamanhos_riscos) else 0)

In [ ]:

print("Perfil dos participantes sem resposta:")

print(
    df_limpo.loc[
        respostas_riscos_pesquisa.isna(),
        'fe_enquadramento'
    ].value_counts(dropna=False)
)

analise_riscos_pesquisa = pd.DataFrame({
    'ID': df_limpo.index + 1,
    'Resposta': respostas_riscos_pesquisa
})

analise_riscos_pesquisa = (
    analise_riscos_pesquisa
    .dropna(subset=['Resposta'])
    .reset_index(drop=True)
)

analise_riscos_pesquisa['Categorias'] = pd.NA

print("\nRespostas para análise:",
      len(analise_riscos_pesquisa))

print("Colunas:",
      analise_riscos_pesquisa.columns.tolist())

display(
    analise_riscos_pesquisa[
        ['ID', 'Categorias']
    ].head()
)

In [ ]:

categorias_riscos_pesquisa = [
    'Autonomia e pensamento crítico',
    'Formação e habilidades acadêmicas',
    'Autoria, plágio e terceirização',
    'Padronização e superficialidade',
    'Erros e referências falsas',
    'Vieses e preconceitos',
    'Criatividade e inovação',
    'Transparência e integridade científica',
    'Privacidade e proteção de dados',
    'Desumanização da pesquisa',
    'Despreparo e falta de regulamentação',
    'Fiscalização e detecção',
    'Produtivismo e pressão acadêmica',
    'Sem opinião ou risco definido',
    'Questionamento do alarmismo'
]

print("Total de categorias:", len(categorias_riscos_pesquisa))

for numero, categoria in enumerate(
    categorias_riscos_pesquisa,
    start=1
):
    print(f"{numero}. {categoria}")

In [ ]:

codigos_riscos_pesquisa = {
    1: [2, 4],
    2: [10],
    3: [1, 2],
    4: [3],
    5: [1],
    6: [1, 2, 3, 4],
    7: [1],
    8: [1, 5, 6, 8],
    9: [1, 7],
    10: [1, 2, 3],

    11: [3, 6, 8, 9],
    12: [1, 10],
    13: [2, 3, 4, 8],
    14: [8],
    15: [1, 4, 7, 9],
    16: [1, 2, 3, 5, 8, 12],
    17: [8],
    18: [1],
    19: [3, 11],
    20: [2, 8],

    21: [3],
    22: [3],
    24: [1, 3],
    25: [1, 2, 8, 10],
    26: [3, 8],
    27: [5],
    28: [1],
    29: [8],
    30: [7],

    31: [4, 7],
    32: [7],
    33: [1, 3, 7, 10],
    34: [5, 6],
    35: [1, 2, 11],
    36: [1, 5],
    37: [1],
    38: [5, 6, 8],
    39: [3],
    41: [1, 2, 5],

    42: [14],
    43: [1, 2, 3, 4, 5],
    44: [1, 2, 7],
    45: [3, 7],
    46: [3],
    47: [1],
    48: [5],
    49: [6, 8],
    50: [12],
    51: [1, 13],

    53: [1, 5],
    54: [1, 3, 4, 5],
    55: [11],
    56: [1, 4],
    57: [3, 4, 5, 13],
    58: [4, 5, 8, 15],
    59: [8],
    60: [1, 5],
    61: [2, 3, 4, 5, 13],
    62: [14],

    63: [4, 5, 6],
    64: [3],
    65: [8],
    66: [1, 4, 5, 7, 9],
    67: [1, 7],
    68: [3, 10],
    69: [3, 5, 6],
    70: [1, 4, 5, 6],
    71: [4]
}

In [ ]:
ids_esperados = set(analise_riscos_pesquisa['ID'])
ids_classificados = set(codigos_riscos_pesquisa)

assert ids_esperados == ids_classificados, (
    "Existem IDs ausentes ou extras no mapeamento."
)

for codigos in codigos_riscos_pesquisa.values():
    assert len(codigos) == len(set(codigos))
    assert all(
        1 <= codigo <= len(categorias_riscos_pesquisa)
        for codigo in codigos
    )

mapa_riscos = {
    id_resposta: [
        categorias_riscos_pesquisa[codigo - 1]
        for codigo in codigos
    ]
    for id_resposta, codigos in codigos_riscos_pesquisa.items()
}

analise_riscos_pesquisa['Categorias'] = (
    analise_riscos_pesquisa['ID'].map(mapa_riscos)
)

print(
    "Respostas classificadas:",
    len(analise_riscos_pesquisa)
)

print(
    "Respostas sem classificação:",
    analise_riscos_pesquisa['Categorias'].isna().sum()
)

print(
    "Respostas com múltiplas categorias:",
    analise_riscos_pesquisa['Categorias']
    .apply(lambda categorias: len(categorias) > 1)
    .sum()
)

print(
    "Total de atribuições temáticas:",
    analise_riscos_pesquisa['Categorias']
    .apply(len)
    .sum()
)

In [ ]:

assert len(analise_riscos_pesquisa) == 68

assert analise_riscos_pesquisa['Categorias'].notna().all()

analise_riscos_pesquisa['Quantidade_categorias'] = (
    analise_riscos_pesquisa['Categorias'].apply(len)
)

print("Respostas classificadas:", len(analise_riscos_pesquisa))

print(
    "Respostas com múltiplas categorias:",
    (analise_riscos_pesquisa['Quantidade_categorias'] > 1).sum()
)

print(
    "Total de atribuições temáticas:",
    analise_riscos_pesquisa['Quantidade_categorias'].sum()
)

print("\nDistribuição de categorias por resposta:")
print(
    analise_riscos_pesquisa['Quantidade_categorias']
    .value_counts()
    .sort_index()
)

sem_opiniao = analise_riscos_pesquisa[
    analise_riscos_pesquisa['Categorias'].apply(
        lambda categorias: 'Sem opinião ou risco definido' in categorias
    )
]

print("\nIDs classificados como sem opinião:")
print(sem_opiniao['ID'].tolist())

inconsistencias = sem_opiniao[
    sem_opiniao['Quantidade_categorias'] > 1
]

print("Inconsistências encontradas:", len(inconsistencias))

assert len(inconsistencias) == 0

In [ ]:

arquivo_riscos_pesquisa = (
    raiz / 'data' / 'categorias_riscos_pesquisa.csv'
)

df_exportacao_riscos = (
    analise_riscos_pesquisa[['ID', 'Categorias']]
    .copy()
)

df_exportacao_riscos['Categorias'] = (
    df_exportacao_riscos['Categorias']
    .apply(lambda categorias: '|'.join(categorias))
)

df_exportacao_riscos.to_csv(
    arquivo_riscos_pesquisa,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo criado:", arquivo_riscos_pesquisa.name)
print("Respostas exportadas:", len(df_exportacao_riscos))
print("Colunas:", df_exportacao_riscos.columns.tolist())

In [ ]:
respostas_beneficios_pesquisa = (
    df_limpo['ab_beneficios_pesquisa']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

print("Total de participantes:", len(df_limpo))

print(
    "Respostas presentes:",
    respostas_beneficios_pesquisa.notna().sum()
)

print(
    "Respostas ausentes:",
    respostas_beneficios_pesquisa.isna().sum()
)

tamanhos_beneficios = (
    respostas_beneficios_pesquisa
    .dropna()
    .str.len()
)

print(
    "\nTamanho médio:",
    round(tamanhos_beneficios.mean(), 1)
    if len(tamanhos_beneficios) else 0
)

print(
    "Menor resposta:",
    tamanhos_beneficios.min()
    if len(tamanhos_beneficios) else 0
)

print(
    "Maior resposta:",
    tamanhos_beneficios.max()
    if len(tamanhos_beneficios) else 0
)

print("\nPerfil dos participantes sem resposta:")

print(
    df_limpo.loc[
        respostas_beneficios_pesquisa.isna(),
        'fe_enquadramento'
    ].value_counts(dropna=False)
)

In [ ]:
analise_beneficios_pesquisa = pd.DataFrame({
    'ID': df_limpo.index + 1,
    'Resposta': respostas_beneficios_pesquisa
})

analise_beneficios_pesquisa = (
    analise_beneficios_pesquisa
    .dropna(subset=['Resposta'])
    .reset_index(drop=True)
)

analise_beneficios_pesquisa['Categorias'] = pd.NA

print("Respostas para análise:", len(analise_beneficios_pesquisa))
print("Colunas:", analise_beneficios_pesquisa.columns.tolist())


In [ ]:
categorias_beneficios_pesquisa = [
    'Agilidade e produtividade',
    'Pesquisa bibliográfica',
    'Síntese e revisão textual',
    'Organização de informações',
    'Análise de dados',
    'Recursos visuais',
    'Apoio à formulação de ideias',
    'Tradução e transcrição',
    'Formatação de trabalhos',
    'Foco na análise crítica',
    'Sem opinião ou benefício definido',
    'Apoio operacional e administrativo',
    'Ampliação metodológica e inovação',
    'Verificação e qualidade das informações'
]

print("Total de categorias:", len(categorias_beneficios_pesquisa))

for numero, categoria in enumerate(
    categorias_beneficios_pesquisa,
    start=1
):
    print(f"{numero}. {categoria}")

In [ ]:
codigos_beneficios_pesquisa = {
    1: [1, 10],
    2: [1],
    3: [2],
    4: [2, 3, 5],
    5: [1],
    6: [11],
    7: [4, 6],
    8: [1, 3, 4, 5, 7, 10],
    9: [4, 6, 9],
    10: [2, 6, 8],

    11: [5, 8],
    12: [4, 12],
    13: [1, 2, 4, 5, 7, 10, 12],
    14: [9],
    15: [1, 3, 12],
    16: [3, 5, 7, 8],
    17: [5],
    18: [1, 4],
    19: [1, 3, 5, 9, 14],
    20: [1, 12],

    21: [3],
    22: [4, 8],
    24: [2, 3, 8],
    25: [1],
    26: [4, 12],
    27: [3, 7, 8],
    28: [2],
    29: [1, 2, 3, 4, 5, 8, 13],
    30: [8],

    31: [1, 3, 4, 6],
    32: [1, 5],
    33: [1, 2, 7, 8],
    34: [2, 4, 5, 8],
    35: [1, 2, 8],
    36: [5],
    37: [11],
    38: [3, 6, 8],
    39: [1, 4],

    41: [1, 2, 13, 14],
    42: [11],
    43: [1, 13],
    44: [2, 4, 12],
    45: [2],
    46: [1],
    47: [1, 2, 8, 12],
    48: [3, 8],
    49: [1, 2, 3, 9],
    50: [5],
    51: [1, 12],

    53: [3],
    54: [2, 8],
    55: [1],
    56: [4],
    57: [2, 4, 6, 10],
    58: [2, 3, 4, 8],
    59: [1, 12],
    60: [1, 4, 5, 8],
    61: [1, 3, 4, 8, 12],
    62: [1],

    63: [1, 2],
    64: [2, 7],
    65: [13],
    66: [1, 2, 4, 6, 12],
    67: [4],
    68: [1, 8],
    69: [2, 3],
    70: [1, 3, 5, 7, 8, 9],
    71: [14]
}

In [ ]:
ids_esperados = set(analise_beneficios_pesquisa['ID'])
ids_classificados = set(codigos_beneficios_pesquisa)

assert ids_esperados == ids_classificados

for codigos in codigos_beneficios_pesquisa.values():
    assert len(codigos) == len(set(codigos))
    assert all(
        1 <= codigo <= len(categorias_beneficios_pesquisa)
        for codigo in codigos
    )

mapa_beneficios = {
    id_resposta: [
        categorias_beneficios_pesquisa[codigo - 1]
        for codigo in codigos
    ]
    for id_resposta, codigos in codigos_beneficios_pesquisa.items()
}

analise_beneficios_pesquisa['Categorias'] = (
    analise_beneficios_pesquisa['ID'].map(mapa_beneficios)
)

print(
    "Respostas classificadas:",
    len(analise_beneficios_pesquisa)
)

print(
    "Respostas sem classificação:",
    analise_beneficios_pesquisa['Categorias'].isna().sum()
)

print(
    "Respostas com múltiplas categorias:",
    analise_beneficios_pesquisa['Categorias']
    .apply(lambda categorias: len(categorias) > 1)
    .sum()
)

print(
    "Total de atribuições temáticas:",
    analise_beneficios_pesquisa['Categorias']
    .apply(len)
    .sum()
)

In [ ]:
analise_beneficios_pesquisa['Quantidade_categorias'] = (
    analise_beneficios_pesquisa['Categorias'].apply(len)
)

print("Distribuição de categorias por resposta:")
print(
    analise_beneficios_pesquisa['Quantidade_categorias']
    .value_counts()
    .sort_index()
)

sem_opiniao = analise_beneficios_pesquisa[
    analise_beneficios_pesquisa['Categorias'].apply(
        lambda categorias:
        'Sem opinião ou benefício definido' in categorias
    )
]

print("\nIDs sem opinião ou benefício definido:")
print(sem_opiniao['ID'].tolist())

inconsistencias = sem_opiniao[
    sem_opiniao['Quantidade_categorias'] > 1
]

print("\nInconsistências:", len(inconsistencias))

assert len(analise_beneficios_pesquisa) == 68
assert analise_beneficios_pesquisa['Quantidade_categorias'].sum() == 169
assert len(inconsistencias) == 0

In [ ]:
arquivo_beneficios = (
    raiz / 'data' / 'categorias_beneficios_pesquisa.csv'
)

df_exportacao_beneficios = (
    analise_beneficios_pesquisa[['ID', 'Categorias']]
    .copy()
)

df_exportacao_beneficios['Categorias'] = (
    df_exportacao_beneficios['Categorias']
    .apply(lambda categorias: '|'.join(categorias))
)

df_exportacao_beneficios.to_csv(
    arquivo_beneficios,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo criado:", arquivo_beneficios.name)
print("Respostas exportadas:", len(df_exportacao_beneficios))
print("Colunas:", df_exportacao_beneficios.columns.tolist())

In [ ]:
respostas_transparencia = (
    df_limpo['ab_diretrizes_transparencia']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

print("Total de participantes:", len(df_limpo))

print(
    "Respostas presentes:",
    respostas_transparencia.notna().sum()
)

print(
    "Respostas ausentes:",
    respostas_transparencia.isna().sum()
)

tamanhos_transparencia = (
    respostas_transparencia
    .dropna()
    .str.len()
)

print(
    "\nTamanho médio:",
    round(tamanhos_transparencia.mean(), 1)
    if len(tamanhos_transparencia) else 0
)

print(
    "Menor resposta:",
    tamanhos_transparencia.min()
    if len(tamanhos_transparencia) else 0
)

print(
    "Maior resposta:",
    tamanhos_transparencia.max()
    if len(tamanhos_transparencia) else 0
)

print("\nPerfil dos participantes sem resposta:")

print(
    df_limpo.loc[
        respostas_transparencia.isna(),
        'fe_enquadramento'
    ].value_counts(dropna=False)
)

In [ ]:
analise_transparencia = pd.DataFrame({
    'ID': df_limpo.index + 1,
    'Resposta': respostas_transparencia
})

analise_transparencia = (
    analise_transparencia
    .dropna(subset=['Resposta'])
    .reset_index(drop=True)
)

analise_transparencia['Categorias'] = pd.NA

print("Respostas para análise:", len(analise_transparencia))
print("Colunas:", analise_transparencia.columns.tolist())


In [ ]:
categorias_transparencia = [
    'Normas e políticas institucionais',
    'Declaração e transparência do uso',
    'Supervisão e responsabilidade humana',
    'Proteção de dados e sigilo',
    'Auditoria, monitoramento e fiscalização',
    'Comunicação e debate',
    'Ética e prevenção de vieses',
    'Padronização de procedimentos',
    'Capacitação e formação',
    'Limites da automação',
    'Ferramentas institucionais e locais',
    'Críticas e dificuldades de controle',
    'Sem opinião ou sugestão',
    'Uso operacional sem diretriz específica'
]

print("Total de categorias:", len(categorias_transparencia))

for numero, categoria in enumerate(
    categorias_transparencia,
    start=1
):
    print(f"{numero}. {categoria}")

In [ ]:
codigos_transparencia = {
    1: [1],
    2: [6],
    3: [2],
    5: [7, 12],
    6: [13],
    7: [6],
    8: [1, 2, 3, 4, 5],
    9: [2, 3],
    10: [2, 3, 4, 5, 7],
    11: [6, 8],

    12: [13],
    13: [1, 3, 4],
    14: [2],
    18: [2],
    19: [12],
    21: [13],
    22: [13],
    27: [2],
    29: [1, 2, 3, 4, 5, 7, 9, 10],
    33: [3, 8],

    36: [13],
    37: [6],
    39: [5],
    40: [13],
    41: [2, 5],
    42: [13],
    43: [1, 9, 11],
    46: [5],
    48: [13],
    50: [13],

    51: [13],
    52: [5, 9],
    54: [10],
    55: [13],
    56: [14],
    58: [13],
    59: [13],
    60: [1, 2, 3, 4, 9, 10],
    61: [13],
    62: [13],

    63: [1, 3, 7],
    64: [1],
    65: [6, 7, 9],
    66: [4],
    67: [10],
    68: [1],
    69: [13]
}

In [ ]:
ids_esperados = set(analise_transparencia['ID'])
ids_classificados = set(codigos_transparencia)

assert ids_esperados == ids_classificados

for codigos in codigos_transparencia.values():
    assert len(codigos) == len(set(codigos))
    assert all(
        1 <= codigo <= len(categorias_transparencia)
        for codigo in codigos
    )

mapa_transparencia = {
    id_resposta: [
        categorias_transparencia[codigo - 1]
        for codigo in codigos
    ]
    for id_resposta, codigos in codigos_transparencia.items()
}

analise_transparencia['Categorias'] = (
    analise_transparencia['ID'].map(mapa_transparencia)
)

print(
    "Respostas classificadas:",
    len(analise_transparencia)
)

print(
    "Respostas sem classificação:",
    analise_transparencia['Categorias'].isna().sum()
)

print(
    "Respostas com múltiplas categorias:",
    analise_transparencia['Categorias']
    .apply(lambda categorias: len(categorias) > 1)
    .sum()
)

print(
    "Total de atribuições temáticas:",
    analise_transparencia['Categorias']
    .apply(len)
    .sum()
)

In [ ]:
analise_transparencia['Quantidade_categorias'] = (
    analise_transparencia['Categorias'].apply(len)
)

print("Respostas classificadas:", len(analise_transparencia))

print(
    "Respostas com múltiplas categorias:",
    (analise_transparencia['Quantidade_categorias'] > 1).sum()
)

print(
    "Total de atribuições temáticas:",
    analise_transparencia['Quantidade_categorias'].sum()
)

print("\nDistribuição de categorias por resposta:")
print(
    analise_transparencia['Quantidade_categorias']
    .value_counts()
    .sort_index()
)

sem_opiniao = analise_transparencia[
    analise_transparencia['Categorias'].apply(
        lambda categorias: 'Sem opinião ou sugestão' in categorias
    )
]

print("\nIDs sem opinião ou sugestão:")
print(sem_opiniao['ID'].tolist())

inconsistencias = sem_opiniao[
    sem_opiniao['Quantidade_categorias'] > 1
]

print("\nInconsistências:", len(inconsistencias))

assert len(analise_transparencia) == 47
assert analise_transparencia['Quantidade_categorias'].sum() == 81
assert len(inconsistencias) == 0

In [ ]:
arquivo_transparencia = (
    raiz / 'data' / 'categorias_transparencia.csv'
)

df_exportacao_transparencia = (
    analise_transparencia[['ID', 'Categorias']]
    .copy()
)

df_exportacao_transparencia['Categorias'] = (
    df_exportacao_transparencia['Categorias']
    .apply(lambda categorias: '|'.join(categorias))
)

df_exportacao_transparencia.to_csv(
    arquivo_transparencia,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo criado:", arquivo_transparencia.name)
print("Respostas exportadas:", len(df_exportacao_transparencia))
print("Colunas:", df_exportacao_transparencia.columns.tolist())

In [ ]:
respostas_politica = (
    df_limpo['ab_comentarios_politica']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

print("Total de participantes:", len(df_limpo))
print("Respostas presentes:", respostas_politica.notna().sum())
print("Respostas ausentes:", respostas_politica.isna().sum())

tamanhos_politica = respostas_politica.dropna().str.len()

print(
    "\nTamanho médio:",
    round(tamanhos_politica.mean(), 1)
    if len(tamanhos_politica) else 0
)

print(
    "Menor resposta:",
    tamanhos_politica.min()
    if len(tamanhos_politica) else 0
)

print(
    "Maior resposta:",
    tamanhos_politica.max()
    if len(tamanhos_politica) else 0
)

print("\nPerfil dos participantes sem resposta:")

print(
    df_limpo.loc[
        respostas_politica.isna(),
        'fe_enquadramento'
    ].value_counts(dropna=False)
)

In [ ]:
analise_politica = pd.DataFrame({
    'ID': df_limpo.index + 1,
    'Resposta': respostas_politica
})

analise_politica = (
    analise_politica
    .dropna(subset=['Resposta'])
    .reset_index(drop=True)
)

analise_politica['Categorias'] = pd.NA

print("Respostas para análise:", len(analise_politica))
print("Colunas:", analise_politica.columns.tolist())


In [ ]:
categorias_politica = [
    'Sem comentário adicional',
    'Sem opinião formada',
    'Apoio à criação da política',
    'Urgência institucional',
    'Capacitação e formação',
    'Participação e debate',
    'Normas e regulamentação',
    'Transparência e fiscalização',
    'Ética e uso crítico',
    'Uso orientado sem proibição generalizada',
    'Adaptação das avaliações acadêmicas',
    'Segurança de dados e tecnologias abertas',
    'Atualização contínua da política',
    'Inclusão digital e desigualdades',
    'Impactos sobre o trabalho humano'
]

print("Total de categorias:", len(categorias_politica))

for numero, categoria in enumerate(categorias_politica, start=1):
    print(f"{numero}. {categoria}")

In [ ]:
codigos_politica = {
    1: [1],
    2: [1],
    6: [5, 10],
    7: [8],
    8: [6, 10],
    9: [1],
    10: [3],
    12: [1],
    13: [1],
    14: [1],
    18: [3],
    19: [1],
    21: [2],
    27: [4, 6],
    29: [7, 8],
    33: [3],
    36: [1],
    37: [1],
    40: [2],
    42: [1],
    43: [5, 9, 12],
    44: [6, 7],
    46: [9, 10],
    47: [7, 11],
    48: [1],
    50: [1],
    51: [1],
    52: [6, 9, 15],
    55: [4],
    56: [1],
    57: [1],
    58: [3],
    59: [9],
    60: [5, 7, 8, 9, 10, 12],
    61: [1],
    62: [2],
    63: [1],
    64: [11],
    65: [13],
    66: [4, 5, 9],
    67: [4, 5, 14],
    68: [3],
    69: [1],
    70: [8],
    71: [1]
}

In [ ]:
ids_esperados = set(analise_politica['ID'])
ids_classificados = set(codigos_politica)

assert ids_esperados == ids_classificados

for codigos in codigos_politica.values():
    assert len(codigos) == len(set(codigos))
    assert all(
        1 <= codigo <= len(categorias_politica)
        for codigo in codigos
    )

mapa_politica = {
    id_resposta: [
        categorias_politica[codigo - 1]
        for codigo in codigos
    ]
    for id_resposta, codigos in codigos_politica.items()
}

analise_politica['Categorias'] = (
    analise_politica['ID'].map(mapa_politica)
)

analise_politica['Quantidade_categorias'] = (
    analise_politica['Categorias'].apply(len)
)

print("Respostas classificadas:", len(analise_politica))

print(
    "Respostas sem classificação:",
    analise_politica['Categorias'].isna().sum()
)

print(
    "Respostas com múltiplas categorias:",
    (analise_politica['Quantidade_categorias'] > 1).sum()
)

print(
    "Total de atribuições temáticas:",
    analise_politica['Quantidade_categorias'].sum()
)

In [ ]:
frequencias_preliminares_politica = (
    analise_politica['Categorias']
    .explode()
    .value_counts()
    .rename_axis('Categoria')
    .reset_index(name='Participantes')
)

frequencias_preliminares_politica['Percentual'] = (
    frequencias_preliminares_politica['Participantes']
    / len(analise_politica) * 100
).round(1)

display(frequencias_preliminares_politica)

assert len(frequencias_preliminares_politica) == 15
assert frequencias_preliminares_politica['Participantes'].sum() == 65

In [ ]:
assert len(analise_politica) == 45
assert analise_politica['ID'].is_unique
assert analise_politica['Categorias'].notna().all()
assert analise_politica['Quantidade_categorias'].sum() == 65

arquivo_politica = raiz / 'data' / 'categorias_politica.csv'

df_exportacao_politica = (
    analise_politica[['ID', 'Categorias']]
    .copy()
)

df_exportacao_politica['Categorias'] = (
    df_exportacao_politica['Categorias']
    .apply(lambda categorias: '|'.join(categorias))
)

df_exportacao_politica.to_csv(
    arquivo_politica,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo criado:", arquivo_politica.name)
print("Respostas exportadas:", len(df_exportacao_politica))
print("Colunas:", df_exportacao_politica.columns.tolist())

In [ ]:
respostas_outros_topicos = (
    df_limpo['ab_outros_topicos']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

print("Total de participantes:", len(df_limpo))
print("Respostas presentes:", respostas_outros_topicos.notna().sum())
print("Respostas ausentes:", respostas_outros_topicos.isna().sum())

tamanhos_outros = respostas_outros_topicos.dropna().str.len()

print(
    "\nTamanho médio:",
    round(tamanhos_outros.mean(), 1)
    if len(tamanhos_outros) else 0
)

print(
    "Menor resposta:",
    tamanhos_outros.min()
    if len(tamanhos_outros) else 0
)

print(
    "Maior resposta:",
    tamanhos_outros.max()
    if len(tamanhos_outros) else 0
)

print("\nPerfil dos participantes sem resposta:")

print(
    df_limpo.loc[
        respostas_outros_topicos.isna(),
        'fe_enquadramento'
    ].value_counts(dropna=False)
)

In [ ]:
analise_outros_topicos = pd.DataFrame({
    'ID': df_limpo.index + 1,
    'Resposta': respostas_outros_topicos
})

analise_outros_topicos = (
    analise_outros_topicos
    .dropna(subset=['Resposta'])
    .reset_index(drop=True)
)

analise_outros_topicos['Categorias'] = pd.NA

print("Respostas para análise:", len(analise_outros_topicos))
print("Colunas:", analise_outros_topicos.columns.tolist())

In [ ]:
categorias_outros_topicos = [
    'Sem tópico adicional',
    'Sem opinião formada',
    'Soberania e desenvolvimento tecnológico',
    'Impactos ambientais',
    'Riscos psicológicos',
    'Confiabilidade e dependência da IA',
    'Capacitação e suporte institucional',
    'Impactos no ensino e nas avaliações',
    'Propriedade intelectual e aspectos jurídicos',
    'Atualização contínua das diretrizes'
]

codigos_outros_topicos = {
    7: [3, 4],
    12: [5],
    14: [3, 6],
    18: [7, 8],
    39: [6],
    47: [8],
    59: [9],
    60: [7, 10],
    67: [7]
}

ids_sem_opiniao_outros = {
    21, 40, 44, 57, 58
}

def classificar_outros_topicos(id_resposta):
    if id_resposta in codigos_outros_topicos:
        return [
            categorias_outros_topicos[codigo - 1]
            for codigo in codigos_outros_topicos[id_resposta]
        ]

    if id_resposta in ids_sem_opiniao_outros:
        return ['Sem opinião formada']

    return ['Sem tópico adicional']

analise_outros_topicos['Categorias'] = (
    analise_outros_topicos['ID']
    .apply(classificar_outros_topicos)
)

In [ ]:
ids_outros = set(analise_outros_topicos['ID'])

assert set(codigos_outros_topicos).issubset(ids_outros)
assert ids_sem_opiniao_outros.issubset(ids_outros)
assert set(codigos_outros_topicos).isdisjoint(
    ids_sem_opiniao_outros
)

frequencias_outros_topicos = (
    analise_outros_topicos['Categorias']
    .explode()
    .value_counts()
    .rename_axis('Categoria')
    .reset_index(name='Participantes')
)

frequencias_outros_topicos['Percentual'] = (
    frequencias_outros_topicos['Participantes']
    / len(analise_outros_topicos) * 100
).round(1)

print("Respostas classificadas:", len(analise_outros_topicos))

print(
    "Respostas com temas específicos:",
    len(codigos_outros_topicos)
)

print(
    "Sem tópico adicional:",
    (frequencias_outros_topicos['Categoria']
     == 'Sem tópico adicional')
    .mul(frequencias_outros_topicos['Participantes'])
    .sum()
)

print(
    "Sem opinião formada:",
    len(ids_sem_opiniao_outros)
)

print(
    "Total de atribuições temáticas:",
    frequencias_outros_topicos['Participantes'].sum()
)

display(frequencias_outros_topicos)

In [ ]:
analise_outros_topicos['Categorias'] = (
    analise_outros_topicos['Categorias']
    .apply(
        lambda categorias: [
            'Sem tópico ou opinião adicional'
            if categoria in [
                'Sem tópico adicional',
                'Sem opinião formada'
            ]
            else categoria
            for categoria in categorias
        ]
    )
)

In [ ]:
frequencias_outros_topicos = (
    analise_outros_topicos['Categorias']
    .explode()
    .value_counts()
    .rename_axis('Categoria')
    .reset_index(name='Participantes')
)

frequencias_outros_topicos['Percentual'] = (
    frequencias_outros_topicos['Participantes']
    / len(analise_outros_topicos) * 100
).round(1)

assert len(analise_outros_topicos) == 41
assert len(frequencias_outros_topicos) == 9
assert frequencias_outros_topicos['Participantes'].sum() == 45

display(frequencias_outros_topicos)

In [ ]:
assert len(analise_outros_topicos) == 41
assert analise_outros_topicos['ID'].is_unique
assert analise_outros_topicos['Categorias'].apply(len).sum() == 45

arquivo_outros_topicos = (
    raiz / 'data' / 'categorias_outros_topicos.csv'
)

df_exportacao_outros = (
    analise_outros_topicos[['ID', 'Categorias']]
    .copy()
)

df_exportacao_outros['Categorias'] = (
    df_exportacao_outros['Categorias']
    .apply(lambda categorias: '|'.join(categorias))
)

df_exportacao_outros.to_csv(
    arquivo_outros_topicos,
    index=False,
    encoding='utf-8-sig'
)

print("Arquivo criado:", arquivo_outros_topicos.name)
print("Respostas exportadas:", len(df_exportacao_outros))

In [ ]:
df_indicadores_orientacoes = pd.read_csv(
    raiz / 'data' / 'indicadores_ich.csv'
)

assert df_indicadores_orientacoes.index.equals(df_limpo.index)

mascara_informais = (
    df_indicadores_orientacoes['orientacao_docente_categoria']
    .eq('Orientações informais')
)

print(
    "Docentes com orientações informais:",
    mascara_informais.sum()
)

colunas_relacionadas = [
    coluna
    for coluna in df_limpo.columns
    if any(
        termo in coluna.lower()
        for termo in [
            'orient',
            'estudant',
            'alun',
            'ensino',
            'pratic'
        ]
    )
]

print("\nColunas relacionadas à orientação dos estudantes:")

for coluna in colunas_relacionadas:
    print(coluna)

In [ ]:
df_informais = (
    df_limpo.loc[
        mascara_informais,
        colunas_relacionadas
    ]
    .copy()
)

df_informais.insert(
    0,
    'ID',
    df_informais.index + 1
)

df_informais.insert(
    1,
    'Familiaridade',
    df_indicadores_orientacoes.loc[
        mascara_informais,
        'fe_familiaridade_ia'
    ].to_numpy()
)

display(df_informais.head(10))